In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "title"),    
    ("##", "chapter"),    
    ("###", "section"),   
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

In [5]:
from dotenv import load_dotenv
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
index_name = "inhouse-python-index"

vector_store = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings,
)


In [6]:
import os
input_dir = './output'

for filename in os.listdir(input_dir):
    if filename.endswith('.md'):
        md_path = os.path.join(input_dir, filename)
        with open(md_path, 'r', encoding='utf-8') as f:
            markdown_text = f.read()
            
        docs = markdown_splitter.split_text(markdown_text)
        for doc in docs:
            doc.metadata['source'] = filename.replace('.md', '')
        vector_store.add_documents(docs)
        
        

In [7]:
retriever = vector_store.as_retriever()

In [11]:
retriever.invoke("대표 이사가 최종 승인해야하는 항목을 전결 규정에서 찾아 리스트업 해줘")

[Document(id='9b118ac3-ef0e-4913-9e8b-9424e09144eb', metadata={'source': 'delegation_of_authority'}, page_content='전결 규정 (Delegation of Authority)\n===============================  \n제 1 장 전결 규정 개요\n------------------  \n제 1조 (목적)\n- 본 규정은 회사 내 의사 결정의 신속성, 책임성을 확보하기 위하여 전결 권한을 명확히 정하고자 한다.  \n제 2조 (적용 범위)\n- 본 규정은 모든 부서 및 직원에게 적용되며, 별도 규정이 있는 경우 해당 규정을 우선으로 한다.  \n제 2 장 전결 권한 분류\n------------------  \n제 3조 (전결 권한 표)  \n| 전결사항                  | 팀장   | 부서장  | 본부장   | 대표이사     |\n|--------------------------|--------|---------|----------|--------------|\n| 인사(채용, 승진, 징계)     | 검토   | 승인    | 승인     | 최종 승인   |\n| 직원 연봉 조정             | 검토   | 승인    | 승인     | 최종 승인   |\n| 연차 및 휴가 승인          | 승인   | 승인    | 보고     | -           |\n| 예산 승인 (50만원 이하)    | 승인   | 승인    | 보고     | -           |\n| 예산 승인 (50만원 초과~200만원) | 검토   | 승인    | 승인     | 최종 승인   |\n| 예산 승인 (200만원 초과)   | 검토   | 검토    | 승인     | 최종 승인   |\n| 출장비 승인               | 승인   | 승인    | 보고     | -           |\n| 법인카드 사용 승인    